# MEDE U12 · CP1: incarcare, aliniere si raport de acoperire
**Unitate:** statie × zi (si statie × ora), Bucuresti, 2025
**Surse:** calitateaer.ro (RNMCA: PM10, NO₂, orar) + Open-Meteo Historical Weather API (temperatura, precipitatii, vant)

Rulare: *Restart & Run All*. Fisierele originale se afla in `data/raw/` si nu se modifica; rezultatele se scriu in `data/processed/`.

In [10]:
from pathlib import Path
import pandas as pd

# radacina repo-ului, indiferent daca notebook-ul ruleaza din notebooks/ sau din radacina
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, OUT = ROOT / "data" / "raw", ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
MIN_HOURS = 18  # zi valida: cel putin 18 din 24 de ore valide

## 1. Maparea statiilor
`loc` = `location_id` din fisierul Open-Meteo. **Verifica maparea** comparand coordonatele din tabelul `meta` (celula urmatoare) cu coordonatele reale ale statiilor de pe calitateaer.ro.

In [11]:
STATIONS = {
    "B-3": {"report": "B3*orar*.xls", "loc": 0},
    "B-4": {"report": "B4*orar*.xls", "loc": 1},
    "B-6": {"report": "B6*orar*.xls", "loc": 2},
}

## 2. Open-Meteo
Fisierul descarcat are timestamp-uri in UTC+3 fix pe tot anul; le convertim la ora locala cu schimbarea la ora de vara, ca sa se potriveasca cu calitateaer. Pastram doar 2025 si transformam vantul din km/h in m/s.

In [12]:
OM_FILE = next(RAW.glob("open-meteo*.xlsx"))
meta = pd.read_excel(OM_FILE, header=0, nrows=3)
print(meta[["location_id", "latitude", "longitude", "elevation"]])

om = pd.read_excel(OM_FILE, header=None, skiprows=6, usecols=range(5),
                   names=["loc", "time", "T", "P", "W"])
om["time"] = (pd.to_datetime(om["time"]) - pd.Timedelta(hours=3)) \
    .dt.tz_localize("UTC").dt.tz_convert("Europe/Bucharest").dt.tz_localize(None)
om = om[(om["time"] >= "2025-01-01") & (om["time"] < "2026-01-01")].copy()
om["W"] = om["W"] / 3.6  # km/h -> m/s
om.groupby("loc").size()

   location_id  latitude  longitude  elevation
0            0   44.4640    26.0890         87
1            1   44.3937    26.1868         83
2            2   44.4640    26.0890         93


,0
loc,
0,8760
1,8760
2,8760


## 3. calitateaer.ro (rapoarte orare)
Etichetele orare 01–24 inseamna sfarsit de ora; le mutam la inceputul orei. Valorile lipsa („-") devin NaN.

In [13]:
def read_rnmca(path):
    d = pd.read_excel(path, header=None, skiprows=11, names=["t", "NO2", "PM10"])
    day, hr = d["t"].str[:10], d["t"].str[11:].astype(int)
    d["time"] = pd.to_datetime(day) + pd.to_timedelta(hr - 1, unit="h")
    for c in ["NO2", "PM10"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    return d.set_index("time")[["NO2", "PM10"]]

## 4. Aliniere: statie × ora si statie × zi

In [14]:
def station_hourly(report_glob, loc):
    path = next(RAW.glob(report_glob))
    m = om[om["loc"] == loc].drop_duplicates("time").set_index("time")[["T", "P", "W"]]
    return read_rnmca(path).join(m, how="inner")

def to_daily(h):
    g = h.groupby(h.index.date)
    out = g.agg(NO2=("NO2", "mean"), PM10=("PM10", "mean"),
                T=("T", "mean"), P=("P", "sum"), W=("W", "max"))
    ok = g[["NO2", "PM10"]].count() >= MIN_HOURS
    out.loc[~ok["NO2"], "NO2"] = float("nan")
    out.loc[~ok["PM10"], "PM10"] = float("nan")
    out.index = pd.to_datetime(out.index)
    return out

hourly = {s: station_hourly(v["report"], v["loc"]) for s, v in STATIONS.items()}
daily = {s: to_daily(h) for s, h in hourly.items()}

hourly_all = pd.concat(hourly, names=["station", "time"]).reset_index()
daily_all = pd.concat(daily, names=["station", "date"]).reset_index()
hourly_all.to_csv(OUT / "hourly_aligned.csv", index=False)
daily_all.to_csv(OUT / "daily_aligned.csv", index=False)
print(hourly_all.shape, daily_all.shape)

(26274, 7) (1095, 7)


## 5. Raport de acoperire
Cate ore si zile supravietuiesc alinierii, pe statie si pe anotimp.

In [15]:
def season(m):
    return {12: "iarna", 1: "iarna", 2: "iarna", 3: "primavara", 4: "primavara", 5: "primavara",
            6: "vara", 7: "vara", 8: "vara"}.get(m, "toamna")

rows = []
for s in STATIONS:
    h, d = hourly[s], daily[s]
    both = d[["NO2", "PM10"]].notna().all(axis=1)
    rows.append({"statie": s, "ore": len(h),
                 "NO2 lipsa %": round(h["NO2"].isna().mean() * 100, 1),
                 "PM10 lipsa %": round(h["PM10"].isna().mean() * 100, 1),
                 "zile valide NO2": int(d["NO2"].notna().sum()),
                 "zile valide PM10": int(d["PM10"].notna().sum()),
                 "zile complete (ambii poluanti)": int(both.sum())})
coverage = pd.DataFrame(rows).set_index("statie")
coverage.to_csv(OUT / "coverage_report.csv")
coverage

,ore,NO2 lipsa %,PM10 lipsa %,zile valide NO2,zile valide PM10,zile complete (ambii poluanti)
statie,,,,,,
B-3,8758,2.0,0.5,355,362,354
B-4,8758,0.0,4.8,365,346,346
B-6,8758,1.1,3.1,359,351,351


In [16]:
dd = daily_all.dropna(subset=["NO2", "PM10"]).copy()
dd["anotimp"] = dd["date"].dt.month.map(season)
by_season = dd.groupby(["station", "anotimp"]).size().unstack()[["iarna", "primavara", "vara", "toamna"]]
by_season.to_csv(OUT / "coverage_by_season.csv")
by_season

anotimp,iarna,primavara,vara,toamna
station,,,,
B-3,87,87,89,91
B-4,77,92,88,89
B-6,90,90,81,90


## 6. Verificari

In [17]:
assert all(h.index.is_unique for h in hourly.values()), "timestamp-uri duplicate"
assert all(len(h) > 8700 for h in hourly.values()), "prea putine ore dupa aliniere"
assert (daily_all.groupby("station").size() == 365).all(), "anul nu are 365 de zile"
print("OK")

OK
